# Part 1: Imports and Configuration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout)

from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint)

import joblib
import random

I0000 00:00:1783391301.561999    5657 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783391304.858994    5657 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783391326.945078    5657 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [81]:
SEED = 42
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

Configuration

In [ ]:
# Approximate Sensor interval

SAMPLE_INTERVAL_SECONDS = 2

# Use Previous 60 seconds
LOOKBACK_SECONDS = 60

#Predict 20 seconds into the future
FORECAST_SECONDS = 20

LOOKBACK = LOOKBACK_SECONDS // SAMPLE_INTERVAL_SECONDS
HORIZON = FORECAST_SECONDS // SAMPLE_INTERVAL_SECONDS

print("Lookback samples: ",LOOKBACK)
print("Forecast Horizon samples: ", HORIZON)

Lookback samples:  30
Forecast Horizon samples:  10


# Part 2: Loading Datasets

In [83]:
FILES = {
    "manan": "manan_merged_experiment.csv",
    "prabhsimrat": "prabh_merged_experiment.csv"
    # "sushant":
}

In [84]:
datasets ={}

for name, path in FILES.items():
    df = pd.read_csv(path)
    df["timestamp"]= pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop= True)
    df["source"] = name
    datasets[name]= df
    print(name, "-", df.shape)

manan - (14803, 13)
prabhsimrat - (15365, 13)


In [85]:
FEATURE_COLS = [
    "cpuUsage",
    "ramUsage",
    "networkConnections",
    "processCount",
    "cpuPackagePower",
    "cpuTemperature"
]

In [86]:
for name, df in datasets.items():

    print(f"\n===== {name} =====")

    print(
        df[
            [
                "cpuUsage",
                "cpuPackagePower",
                "cpuAverageClock",
                "cpuTemperature"
            ]
        ].describe().loc[
            ["min", "mean", "std", "max"]
        ]
    )


===== manan =====
        cpuUsage  cpuPackagePower  cpuAverageClock  cpuTemperature
min     9.200000         9.600000       403.200000       60.000000
mean   58.584071        40.438654      2419.553636       86.093292
std    30.660576        15.644110       471.266927       11.716394
max   100.000000        66.600000      2973.600000       99.000000

===== prabhsimrat =====
        cpuUsage  cpuPackagePower  cpuAverageClock  cpuTemperature
min     1.000000         6.000000      2810.000000       64.100000
mean   49.854247        29.637592      3617.252652       87.949492
std    34.927009        11.036970       287.802921       10.251866
max   100.000000        50.900000      4492.000000       95.900000


# Part 3: Cleaning 

After running (after_stress_generator.ipynb)- most of the cleaning is already done

In [87]:
def clean_run(df):
    df = df.copy()
    # Keep only valid sensor data
    df = df.dropna(subset = FEATURE_COLS)
    df = df.sort_values(
        "timestamp"
    ).reset_index(drop=True)

    return df

In [88]:
for name in datasets:

    datasets[name] = clean_run(
        datasets[name]
    )

    print(
        name,
        len(datasets[name])
    )

manan 14803
prabhsimrat 15365


# Checking Sampling Gaps

In [89]:
def inspect_time_gaps(df, name):

    gaps = (
        df["timestamp"]
        .diff()
        .dt.total_seconds()
    )

    print(f"\n{name}")

    print(
        "Median interval:",
        gaps.median()
    )

    print(
        "Maximum gap:",
        gaps.max()
    )

    print(
        "Gaps > 5 seconds:",
        (gaps > 5).sum()
    )

In [90]:
for name, df in datasets.items():

    inspect_time_gaps(
        df,
        name
    )


manan
Median interval: 2.0298681
Maximum gap: 4.3050749
Gaps > 5 seconds: 0

prabhsimrat
Median interval: 2.02355955
Maximum gap: 7.0642233
Gaps > 5 seconds: 1


# Part 6: Create Balanced Time Blocks 

In [91]:
TRAIN_RUNS = ["manan", "prabhsimrat"]
TEST_RUNS = ["manan", "prabhsimrat"]

In [92]:
BLOCK_MINUTES = 10

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

In [93]:
def get_phase_group(mode):

    mode = str(mode).upper()

    if "COOLING" in mode:
        return "COOLING"

    if mode in [
        "INITIAL_IDLE",
        "PRE_EXPERIMENT",
        "POST_EXPERIMENT"
    ]:
        return "IDLE"

    if mode in [
        "RAMP",
        "CHAOS",
        "TRANSITION",
        "MIXED"
    ]:
        return mode

    return "OTHER"

In [94]:
for name, df in datasets.items():

    df = df.copy()

    df["phaseGroup"] = (
        df["mode"]
        .apply(get_phase_group)
    )

    datasets[name] = df

    print(f"\n===== {name} =====")

    print(
        df["phaseGroup"]
        .value_counts()
    )


===== manan =====
phaseGroup
MIXED         5298
CHAOS         2658
RAMP          2656
TRANSITION    2654
OTHER          739
COOLING        739
IDLE            59
Name: count, dtype: int64

===== prabhsimrat =====
phaseGroup
MIXED         5306
CHAOS         2669
RAMP          2664
TRANSITION    2656
OTHER         1574
COOLING        445
IDLE            51
Name: count, dtype: int64


Now Creating Contiguous Blocks

In [95]:
def create_time_blocks(df, run_name, block_minutes = 10):
    df = df.copy()
    df = df.sort_values("timestamp").reset_index(drop= True)
    all_blocks = []
    block_counter = 0
    
    # --------------------- Processing Each Contiguous Phase Separately----------------------------
    
    phase_change = (df["phaseGroup"]!=df["phaseGroup"].shift())
    
    df["phaseSegment"]= (phase_change.cumsum())
    
    for segment_id, segment in df.groupby("phaseSegment"):
        segment = (segment.sort_values("timestamp").copy())
        phase = (segment["phaseGroup"].iloc[0])
        segment_start = (segment["timestamp"].min())
        
        # Assign 10 minute block number
        elapsed_minutes = (segment["timestamp"] - segment_start).dt.total_seconds() / 60
        
        segment["localBlock"] = (elapsed_minutes // block_minutes).astype(int)
        
        for local_block, block in segment.groupby("localBlock"):
            block = block.copy()
            block["blockId"] = (f"{run_name}"
                                f"{block_counter}")
            block["runName"] = run_name
            all_blocks.append(block)
            block_counter+=1
    
    return all_blocks

In [96]:
all_blocks = []

for name in TRAIN_RUNS:

    run_blocks = create_time_blocks(

        df=datasets[name],

        run_name=name,

        block_minutes=BLOCK_MINUTES
    )

    all_blocks.extend(
        run_blocks
    )

    print(
        name,
        "blocks:",
        len(run_blocks)
    )

manan blocks: 54
prabhsimrat blocks: 55


# 7. Remove Blocks that are too short

In [97]:
MIN_REQUIRED_SAMPLES = (
    LOOKBACK
    + HORIZON
    + 5
)

In [98]:
valid_blocks = []

removed_blocks = []

for block in all_blocks:

    if len(block) >= MIN_REQUIRED_SAMPLES:

        valid_blocks.append(block)

    else:

        removed_blocks.append(block)


print(
    "Valid blocks:",
    len(valid_blocks)
)

print(
    "Removed short blocks:",
    len(removed_blocks)
)

Valid blocks: 108
Removed short blocks: 1


In [99]:
block_summary = []

for block in valid_blocks:

    block_summary.append({

        "blockId":
            block["blockId"].iloc[0],

        "runName":
            block["runName"].iloc[0],

        "phaseGroup":
            block["phaseGroup"].iloc[0],

        "samples":
            len(block),

        "startTime":
            block["timestamp"].min(),

        "endTime":
            block["timestamp"].max()
    })


block_summary = pd.DataFrame(
    block_summary
)

display(
    block_summary.head(20)
)

,blockId,runName,phaseGroup,samples,startTime,endTime
0,manan1,manan,IDLE,59,2026-07-06 02:15:52.811050700+05:30,2026-07-06 02:17:50.805294900+05:30
1,manan2,manan,RAMP,296,2026-07-06 02:17:52.834077200+05:30,2026-07-06 02:27:51.946574700+05:30
2,manan3,manan,RAMP,295,2026-07-06 02:27:53.976796400+05:30,2026-07-06 02:37:52.628830900+05:30
3,manan4,manan,RAMP,295,2026-07-06 02:37:54.652708500+05:30,2026-07-06 02:47:51.481890400+05:30
4,manan5,manan,RAMP,296,2026-07-06 02:47:53.518745400+05:30,2026-07-06 02:57:51.959968500+05:30
5,manan6,manan,RAMP,292,2026-07-06 02:57:54.002098900+05:30,2026-07-06 03:07:50.885778200+05:30
6,manan7,manan,RAMP,296,2026-07-06 03:07:52.939946600+05:30,2026-07-06 03:17:51.375515200+05:30
7,manan8,manan,RAMP,296,2026-07-06 03:17:53.410543100+05:30,2026-07-06 03:27:52.352622600+05:30
8,manan9,manan,RAMP,295,2026-07-06 03:27:54.375094400+05:30,2026-07-06 03:37:52.734991300+05:30
9,manan10,manan,RAMP,295,2026-07-06 03:37:54.764023700+05:30,2026-07-06 03:47:51.479478400+05:30


In [100]:
print(
    block_summary[
        "phaseGroup"
    ].value_counts()
)

phaseGroup
MIXED         36
RAMP          18
CHAOS         18
TRANSITION    18
OTHER          9
COOLING        7
IDLE           2
Name: count, dtype: int64


# 8. Split whole blocks into train, validation and internal test

In [101]:
from sklearn.model_selection import train_test_split

In [102]:
def split_blocks_balanced(
    block_summary,
    seed=42
):

    train_ids = []
    val_ids = []
    test_ids = []

    # ----------------------------------------
    # Split separately by:
    # device + experiment phase
    # ----------------------------------------

    grouped = block_summary.groupby(
        [
            "runName",
            "phaseGroup"
        ]
    )


    for (
        run_name,
        phase
    ), group in grouped:

        ids = (
            group["blockId"]
            .tolist()
        )


        # Reproducible random order
        rng = np.random.default_rng(
            seed
        )

        rng.shuffle(ids)


        n = len(ids)


        print(
            run_name,
            phase,
            "blocks:",
            n
        )


        # ------------------------------------
        # Very small groups
        # ------------------------------------

        if n == 1:

            train_ids.extend(ids)

            continue


        if n == 2:

            train_ids.append(ids[0])
            val_ids.append(ids[1])

            continue


        # ------------------------------------
        # Normal groups
        # ------------------------------------

        n_train = max(
            1,
            int(round(n * TRAIN_RATIO))
        )

        n_val = max(
            1,
            int(round(n * VAL_RATIO))
        )


        # Ensure at least one test block
        if (
            n_train
            + n_val
            >= n
        ):

            n_train = n - 2
            n_val = 1
            

        train_ids.extend(
            ids[:n_train]
        )


        val_ids.extend(
            ids[
                n_train:
                n_train + n_val
            ]
        )


        test_ids.extend(
            ids[
                n_train + n_val:
            ]
        )


    return (
        train_ids,
        val_ids,
        test_ids
    )

In [103]:
(
    train_block_ids,
    val_block_ids,
    test_block_ids

) = split_blocks_balanced(
    block_summary,
    seed=SEED
)

manan CHAOS blocks: 9
manan COOLING blocks: 4
manan IDLE blocks: 1
manan MIXED blocks: 18
manan OTHER blocks: 3
manan RAMP blocks: 9
manan TRANSITION blocks: 9
prabhsimrat CHAOS blocks: 9
prabhsimrat COOLING blocks: 3
prabhsimrat IDLE blocks: 1
prabhsimrat MIXED blocks: 18
prabhsimrat OTHER blocks: 6
prabhsimrat RAMP blocks: 9
prabhsimrat TRANSITION blocks: 9


In [104]:
print(
    "\nTraining blocks:",
    len(train_block_ids)
)

print(
    "Validation blocks:",
    len(val_block_ids)
)

print(
    "Internal test blocks:",
    len(test_block_ids)
)


Training blocks: 72
Validation blocks: 16
Internal test blocks: 20


In [105]:
block_summary["split"] = (
    "UNASSIGNED"
)

block_summary.loc[

    block_summary["blockId"]
    .isin(train_block_ids),

    "split"

] = "TRAIN"


block_summary.loc[

    block_summary["blockId"]
    .isin(val_block_ids),

    "split"

] = "VALIDATION"


block_summary.loc[

    block_summary["blockId"]
    .isin(test_block_ids),

    "split"

] = "TEST"

In [106]:
coverage = pd.crosstab(

    block_summary["phaseGroup"],

    block_summary["split"]
)

display(coverage)

split,TEST,TRAIN,VALIDATION
phaseGroup,,,
CHAOS,4,12,2
COOLING,2,3,2
IDLE,0,2,0
MIXED,4,26,6
OTHER,2,5,2
RAMP,4,12,2
TRANSITION,4,12,2


In [107]:
device_coverage = pd.crosstab(

    block_summary["runName"],

    block_summary["split"]
)

display(device_coverage)

split,TEST,TRAIN,VALIDATION
runName,,,
manan,10,35,8
prabhsimrat,10,37,8


# 10. Converting IDs back to block lists

In [108]:
train_blocks = []

val_blocks = []

test_blocks = []


for block in valid_blocks:

    block_id = (
        block["blockId"]
        .iloc[0]
    )


    if block_id in train_block_ids:

        train_blocks.append(
            block
        )


    elif block_id in val_block_ids:

        val_blocks.append(
            block
        )


    elif block_id in test_block_ids:

        test_blocks.append(
            block
        )

In [109]:
print(
    "Train blocks:",
    len(train_blocks)
)

print(
    "Validation blocks:",
    len(val_blocks)
)

print(
    "Internal test blocks:",
    len(test_blocks)
)

Train blocks: 72
Validation blocks: 16
Internal test blocks: 20


# Part 11: Fitting the scaler on traininig blocks only

In [110]:
scaler_training_data = pd.concat(

    [
        block[FEATURE_COLS]
        for block in train_blocks
    ],

    ignore_index=True
)

In [111]:
scaler_X = StandardScaler()

scaler_X.fit(
    scaler_training_data
)

StandardScaler()

In [112]:
joblib.dump(

    scaler_X,

    "cross_device_feature_scaler.pkl"
)

print(
    "Scaler fitted only on training blocks."
)

Scaler fitted only on training blocks.


# Part 12: Create sequences inside each block

In [113]:
def create_sequences_from_block(
    block,
    scaler,
    lookback,
    horizon,
    max_gap_seconds=5
):

    block = (
        block
        .sort_values("timestamp")
        .reset_index(drop=True)
        .copy()
    )


    scaled_features = scaler.transform(
        block[FEATURE_COLS]
    )


    temperatures = (

        block["cpuTemperature"]

        .to_numpy()
    )


    timestamps = (

        block["timestamp"]

        .to_numpy()
    )


    X = []
    y = []

    current_temperatures = []

    future_temperatures = []

    prediction_timestamps = []


    for i in range(

        lookback,

        len(block) - horizon + 1
    ):


        sequence_start = (
            i - lookback
        )


        current_index = (
            i - 1
        )


        future_index = (

            current_index

            + horizon
        )


        if future_index >= len(block):

            break


        # ------------------------------------
        # Check entire history → future period
        # ------------------------------------

        relevant_times = (

            block["timestamp"]

            .iloc[
                sequence_start:
                future_index + 1
            ]
        )


        gaps = (

            relevant_times

            .diff()

            .dt.total_seconds()

            .dropna()
        )


        if (
            gaps > max_gap_seconds
        ).any():

            continue


        # ------------------------------------
        # Input
        # ------------------------------------

        X.append(

            scaled_features[
                sequence_start:i
            ]
        )


        # ------------------------------------
        # Target
        # ------------------------------------

        current_temp = (

            temperatures[
                current_index
            ]
        )


        future_temp = (

            temperatures[
                future_index
            ]
        )


        delta_temp = (

            future_temp

            - current_temp
        )


        y.append(
            delta_temp
        )


        current_temperatures.append(
            current_temp
        )


        future_temperatures.append(
            future_temp
        )


        prediction_timestamps.append(
            timestamps[
                future_index
            ]
        )


    return (

        np.asarray(X),

        np.asarray(y),

        np.asarray(
            current_temperatures
        ),

        np.asarray(
            future_temperatures
        ),

        np.asarray(
            prediction_timestamps
        )
    )

# 13. Create datasets from block lists

In [114]:
def create_dataset_from_blocks(
    blocks,
    scaler
):

    all_X = []
    all_y = []

    all_current = []
    all_future = []

    all_timestamps = []

    all_phases = []
    all_runs = []


    for block in blocks:


        (
            X,
            y,
            current,
            future,
            timestamps

        ) = create_sequences_from_block(

            block=block,

            scaler=scaler,

            lookback=LOOKBACK,

            horizon=HORIZON
        )


        if len(X) == 0:

            continue


        phase = (
            block[
                "phaseGroup"
            ].iloc[0]
        )


        run_name = (
            block[
                "runName"
            ].iloc[0]
        )


        all_X.append(X)

        all_y.append(y)

        all_current.append(
            current
        )

        all_future.append(
            future
        )

        all_timestamps.append(
            timestamps
        )


        all_phases.extend(

            [phase] * len(X)
        )


        all_runs.extend(

            [run_name] * len(X)
        )


    return (

        np.concatenate(all_X),

        np.concatenate(all_y),

        np.concatenate(all_current),

        np.concatenate(all_future),

        np.concatenate(all_timestamps),

        np.asarray(all_phases),

        np.asarray(all_runs)
    )

In [115]:
(
    X_train,
    y_train,
    train_current,
    train_actual,
    train_timestamps,
    train_phases,
    train_runs

) = create_dataset_from_blocks(

    train_blocks,

    scaler_X
)

In [116]:
(
    X_val,
    y_val,
    val_current,
    val_actual,
    val_timestamps,
    val_phases,
    val_runs

) = create_dataset_from_blocks(

    val_blocks,

    scaler_X
)

In [117]:
(
    X_test,
    y_test,
    test_current,
    test_actual,
    test_timestamps,
    test_phases,
    test_runs

) = create_dataset_from_blocks(

    test_blocks,

    scaler_X
)

In [118]:
print("\nTRAIN")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("\nVALIDATION")
print("X:", X_val.shape)
print("y:", y_val.shape)

print("\nINTERNAL TEST")
print("X:", X_test.shape)
print("y:", y_test.shape)


TRAIN
X: (17273, 30, 6)
y: (17273,)

VALIDATION
X: (3810, 30, 6)
y: (3810,)

INTERNAL TEST
X: (4829, 30, 6)
y: (4829,)


# 14. Verify Actual Sequence Diversity

In [119]:
def show_sequence_distribution(
    phases,
    name
):

    print(
        f"\n===== {name} ====="
    )

    print(

        pd.Series(phases)

        .value_counts()
    )

In [120]:
show_sequence_distribution(
    train_phases,
    "TRAIN"
)

show_sequence_distribution(
    val_phases,
    "VALIDATION"
)

show_sequence_distribution(
    test_phases,
    "INTERNAL TEST"
)


===== TRAIN =====
MIXED         6641
CHAOS         3084
RAMP          3080
TRANSITION    3032
OTHER          929
COOLING        475
IDLE            32
Name: count, dtype: int64

===== VALIDATION =====
MIXED         1541
CHAOS          513
TRANSITION     513
OTHER          513
RAMP           512
COOLING        218
Name: count, dtype: int64

===== INTERNAL TEST =====
CHAOS         1028
RAMP          1026
TRANSITION    1024
MIXED         1018
OTHER          515
COOLING        218
Name: count, dtype: int64


# 15. Building the LSTM

In [121]:
model = Sequential([
    Input(shape=(LOOKBACK, len(FEATURE_COLS))),

    LSTM(32),

    Dropout(0.20),

    Dense(16, activation="relu"),

    Dense(1)
])

In [122]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss=tf.keras.losses.Huber(delta=2.0),
    metrics=["mae"]
)

# Part 16: Train

In [123]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=12,
    min_delta=0.01,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

Train

In [124]:
history = model.fit(

    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,

    batch_size=64,

    shuffle=True,

    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ],

    verbose=1
)

Epoch 1/100
267/270 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 4.1824 - mae: 2.6832
Epoch 1: val_loss improved from 49.31575 to 3.50990, saving model to best_cross_device_lstm.keras

Epoch 1: finished saving model to best_cross_device_lstm.keras
270/270 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 4.1750 - mae: 2.6796 - val_loss: 3.5099 - val_mae: 2.3386 - learning_rate: 5.0000e-04
Epoch 2/100
266/270 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 4.1496 - mae: 2.6763
Epoch 2: val_loss improved from 3.50990 to 3.49116, saving model to best_cross_device_lstm.keras

Epoch 2: finished saving model to best_cross_device_lstm.keras
270/270 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 4.1427 - mae: 2.6729 - val_loss: 3.4912 - val_mae: 2.3244 - learning_rate: 5.0000e-04
Epoch 3/100
268/270 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 4.1299 - mae: 2.6673
Epoch 3: val_loss improved from 3.49116 to 3.48144, saving model to best_cross_device_lstm.keras

Epoch 3: finished saving model to best_cross_device_lstm.ker

In [136]:
def calculate_metrics(actual, predicted, name):

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    r2 = r2_score(
        actual,
        predicted
    )

    print(f"\n===== {name} =====")
    print(f"MAE:  {mae:.3f} °C")
    print(f"RMSE: {rmse:.3f} °C")
    print(f"R²:   {r2:.4f}")

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [137]:
    # Load the actual best checkpoint explicitly
best_model = tf.keras.models.load_model(
    "best_cross_device_lstm.keras"
)

# Predict temperature change
test_predicted_delta = (
    best_model.predict(X_test)
    .flatten()
)

# Reconstruct future temperature
test_lstm_predictions = (
    test_current
    + test_predicted_delta
)

# Persistence baseline:
# future temperature = current temperature
test_persistence = test_current.copy()

151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [138]:
test_predicted_delta = (

    model.predict(
        X_test
    )

    .flatten()
)

151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [139]:
test_lstm_predictions = (

    test_current

    +

    test_predicted_delta
)

In [140]:
test_persistence = (

    test_current.copy()
)

In [141]:
internal_baseline = calculate_metrics(

    test_actual,

    test_persistence,

    "Internal Test - Persistence"
)


===== Internal Test - Persistence =====
MAE:  2.799 °C
RMSE: 5.614 °C
R²:   0.7644


In [142]:

internal_lstm = calculate_metrics(
    test_actual,
    test_lstm_predictions,
    "Internal Test - LSTM"
)


===== Internal Test - LSTM =====
MAE:  2.810 °C
RMSE: 5.466 °C
R²:   0.7767


In [143]:
def evaluate_by_phase(

    actual,
    predicted,
    phases,
    model_name
):

    results = []


    for phase in np.unique(
        phases
    ):

        mask = (
            phases == phase
        )


        if mask.sum() < 10:

            continue


        mae = mean_absolute_error(

            actual[mask],

            predicted[mask]
        )


        rmse = np.sqrt(

            mean_squared_error(

                actual[mask],

                predicted[mask]
            )
        )


        results.append({

            "Model":
                model_name,

            "Phase":
                phase,

            "Samples":
                mask.sum(),

            "MAE":
                mae,

            "RMSE":
                rmse
        })


    return pd.DataFrame(
        results
    )

In [144]:
lstm_phase_results = evaluate_by_phase(

    test_actual,

    test_lstm_predictions,

    test_phases,

    "LSTM"
)

In [145]:
baseline_phase_results = evaluate_by_phase(

    test_actual,

    test_persistence,

    test_phases,

    "Persistence"
)

In [146]:
phase_comparison = pd.concat(

    [
        baseline_phase_results,
        lstm_phase_results
    ],

    ignore_index=True
)

display(
    phase_comparison
)

,Model,Phase,Samples,MAE,RMSE
0,Persistence,CHAOS,1028,3.928891,7.352621
1,Persistence,COOLING,218,2.396789,3.580701
2,Persistence,MIXED,1018,3.310118,6.830900
3,Persistence,OTHER,515,2.911650,3.972769
4,Persistence,RAMP,1026,1.652534,3.224067
5,Persistence,TRANSITION,1024,2.334277,5.183320
6,LSTM,CHAOS,1028,3.925678,7.122195
7,LSTM,COOLING,218,2.511707,3.513100
8,LSTM,MIXED,1018,3.368873,6.664484
9,LSTM,OTHER,515,2.958883,3.971527
